### Introduction

#### 1. What the authors are pointing out

In earlier chapters (like the ones you've been studying in D2L: linear regression, softmax, etc.), the workflow looked like this:
1. You have a dataset.
2. Train a model.
3. Evaluate it on a test set.
4. If the test accuracy is good → the model is considered successful.

But the authors are saying:
In real applications, this workflow ignores two crucial questions.

Two missing questions:
1. Where did the data come from?
2. What will happen when the model is actually used?

In practice, many ML projects fail because these questions are ignored.

#### 2. Problem 1: Distribution shift

The first issue mentioned is data distribution shift.
The text says:

Sometimes models appear to perform marvelously on the test set but fail in deployment when the distribution of data suddenly shifts.

Meaning:
- Training data distribution:

$$
P_{\text {train }}(x, y)
$$

- Real-world deployment distribution:

$$
P_{\text {deploy }}(x, y)
$$


If

$$
P_{\text {train }} \neq P_{\text {deploy }}
$$

then the model can fail.
Example:
A model trained on:
- historical loan applications
- historical borrower behavior

But the future population might be different.

Typical examples in practice:
- self-driving cars trained in sunny weather → fail in snow
- speech models trained on adults → used by children
- financial models trained on past markets → future regime changes

#### 3. Problem 2: The moel changes the environment

The second issue is even more subtle.
Sometimes the model itself changes the data distribution.
The text gives a deliberately silly example:
Example: Shoe-based credit model
Suppose the model learns:

* Oxfords -> repay loan
* Sneakers -> default

So the bank decides:
- approve loans $\boldsymbol{\rightarrow}$ Oxfords
- deny loans → Sneakers

But once people notice this rule:
Everyone will start wearing Oxfords.

Then:

* Oxford -> no longer indictaes creditworthiness

So the original correlation disappears.
The model breaks because it changed behavior.

### Type of Distribution Shift

The authors first set up a general framework.
Training data:

$$
(\mathrm{x}, y) \sim p_S(\mathrm{x}, y)
$$


Test data:

$$
(\mathrm{x}, y) \sim p_T(\mathrm{x}, y)
$$

- $S=$ source distribution (training)
- T= target distribution (test)

The key issue:

$$
p_S(\mathbf{x}, y) \neq p_T(\mathbf{x}, y)
$$

Why this is a problem
If we make no assumptions about how they differ, learning is impossible.
The note gives a pathological example:

$$
p_S(\mathbf{x})=p_T(\mathbf{x})
$$

but

$$
p_S(y \mid \mathbf{x})=1-p_T(y \mid \mathbf{x})
$$


Meaning:
- the inputs are identical
- but the labels flip

Example:

| Image      | Training label | Test label |
|------------|---------------|------------|
| Cat image  | cat           | dog        |
| Dog image  | dog           | cat        |

If this happened, no algorithm could detect the change, because the inputs look identical.
So we must assume something about how the distribution shifts.
The rest of the section describes three structured assumptions.

#### 1. Covariate Shift

Definition
Covariate shift assumes:

$$
P_S(y \mid \mathbf{x})=P_T(y \mid \mathbf{x})
$$

but

$$
P_S(\mathrm{x}) \neq P_T(\mathrm{x})
$$


Meaning:
- the labeling rule stays the same
- the input distribution changes

Example in the note

Training data: photos

Test data: cartoons

Your training set looks like this:
(photos of cats and dogs)

But the test set looks like this:
(cartoon drawings of cats and dogs)

The appearance distribution changed:

$$
P_S(\mathbf{x}) \neq P_T(\mathbf{x})
$$


But the concept cat vs dog is the same:

$$
P(y \mid \mathbf{x}) \text { unchanged }
$$

Intuition
The classifier you learned is still correct in principle, but:
- it has never seen this input style

So performance drops.

Real-world examples
Common in ML:
- Autonomous driving
    - train: sunny
    - test: snow
- Medical imaging
    - train: one hospital
    - test: another hospital
- Speech recognition
    - train: studio audio
    - test: noisy phone calls

#### 2. Label Shift

Now we flip the assumption.
Definition
Label shift assumes:

$$
P_S(\mathbf{x} \mid y)=P_T(\mathbf{x} \mid y)
$$

but

$$
P_S(y) \neq P_T(y)
$$


Meaning:
- the class proportions change
- the appearance of each class stays the same

Example
Suppose we classify diseases.
Training data:

| Disease | Probability |
| :--- | :--- |
| Flu | 80% |
| COVID | 20% |
| Later: |  |
| Disease | Probability |
| Flu | 40% |
| COVID | 60% |

Symptoms of each disease remain the same:

$$
P(\mathbf{x} \mid y) \text { unchanged }
$$


But prevalence changes:

$$
P(y) \text { changed }
$$



Why this assumption makes sense

The authors say:
label shift is appropriate when $y$ causes $x$

Example:
disease → symptoms

So if the disease distribution changes, symptoms change indirectly.

Why label shift is easier to handle
The note makes a subtle point.
Label-shift corrections often involve manipulating:

$$
P(y)
$$

which is low dimensional.
While covariate shift involves:

$$
P(x)
$$

which is high dimensional (especially images).
So label shift is often easier to correct.

#### 3. Concept Shift

Now we reach the most difficult case.

Definition

Concept shift means:

$$
P(y \mid x) \text { changes }
$$


Meaning: the labeling rule itself changes.

Example in the note
Soft drink naming in the US.
Different regions say:

| Region | Word |
| :--- | :--- |
| Northeast | soda |
| Midwest | pop |
| South | coke |

Same object → different label.

So:
$P(y \mid x)$ varies geographically

Other examples
Concept shift happens when definitions evolve.
Examples:
- medical diagnostic criteria
- job titles
- slang language
- fashion

For instance:
"Al engineer" meant something very different 15 years ago.

Why concept shift is difficult

Because the mapping itself changes.

old rule: $f(x)$

new rule: $f^{\prime}(x)$

So the model must continually adapt.

One more insight (very important)
Notice something interesting:

$$
P(x, y)=P(y \mid x) P(x)=P(x \mid y) P(y)
$$


So distribution shift can arise from three components:
1. change in $P(x) \rightarrow$ covariate shift
2. change in $P(y) \rightarrow$ label shift
3. change in $P(y \mid x) \rightarrow$ concept shift

### Correction of Distribution Shift

#### Empirical Risk vs True Risk

During training we minimize

$$
\frac{1}{n} \sum_{i=1}^n l\left(f\left(x_i\right), y_i\right)
$$


This is called empirical risk.
Empirical Risk

$$
\hat{R}(f)=\frac{1}{n} \sum_{i=1}^n l\left(f\left(x_i\right), y_i\right)
$$


It is just the average loss on the training data.

True Risk (Population Risk)
What we really want is:

$$
R(f)=E_{p(x, y)}[l(f(x), y)]
$$

or written as an integral

$$
\iint l(f(x), y) p(x, y) d x d y
$$


This is the expected loss over the true data distribution.

Why we minimize empirical risk

Because we cannot access the full population distribution $p(x, y)$.

So we approximate it with the dataset.

This idea is called:

Empirical Risk Minimization (ERM)

This is the foundation of most machine learning.

Why distribution shift breaks ERM
ERM assumes:

$$
p_{\text {train }}(x, y)=p_{\text {test }}(x, y)
$$


But if

$$
q(x, y) \neq p(x, y)
$$

then minimizing training loss may not minimize test loss.

The rest of the section explains how to fix this.

#### Covariate Shift Correction

Recall the covariate shift assumption:

$$
p(y \mid x)=q(y \mid x)
$$

but

$$
p(x) \neq q(x)
$$


Meaning:
- the labeling rule is unchanged
- the input distribution changes

Key trick: importance weighting
We start from the true risk

$$
R(f)=\iint l(f(x), y) p(y \mid x) p(x) d x d y
$$


Now substitute

$$
p(y \mid x)=q(y \mid x)
$$


Then multiply and divide by $q(x)$ :

$$
=\iint l(f(x), y) q(y \mid x) q(x) \frac{p(x)}{q(x)} d x d y
$$


So the expectation becomes:

$$
E_{q(x, y)}\left[\frac{p(x)}{q(x)} l(f(x), y)\right]
$$

Importance weights
Define

$$
\beta(x)=\frac{p(x)}{q(x)}
$$


Then training objective becomes

$$
\frac{1}{n} \sum_{i=1}^n \beta_i l\left(f\left(x_i\right), y_i\right)
$$


So instead of normal training we do
Weighted empirical risk minimization

Intuition
If a datapoint is more common in the test distribution, we weight it more.
If a datapoint is rare in the test distribution, we weight it less.

Problem: we don't know $p(x) / q(x)$

So we must estimate it.

The clever trick is:

Train a classifier to distinguish training vs test data

Logistic regression trick
Create a dataset:

| x | label |
| :--- | :--- |
| training sample | -1 |
| test sample | +1 |

Train a classifier predicting
$$P(z=1 \mid x)$$
where
- $z=1 \rightarrow$ test
- $z=-1 \rightarrow$ training

Then

$$
P(z=1 \mid x)=\frac{p(x)}{p(x)+q(x)}
$$


From this we get

$$
\frac{P(z=1 \mid x)}{P(z=-1 \mid x)}=\frac{p(x)}{q(x)}
$$


So the importance weight is the odds ratio.

If logistic regression outputs

$$
P(z=1 \mid x)=\frac{1}{1+e^{-h(x)}}
$$

then

$$
\beta(x)=e^{h(x)}
$$

Final algorithm
1. Combine training and test features
2. Train classifier: training vs test
3. Compute weights
$$
\beta_i=e^{h\left(x_i\right)}
$$
4. Train original model using weighted loss

#### Label Shift Correction

Now assume

$$
p(x \mid y)=q(x \mid y)
$$

but

$$
p(y) \neq q(y)
$$


Meaning:
- class proportions change
- class appearance stays same

Importance weighting again

The risk becomes

$$
\iint l(f(x), y) q(x \mid y) q(y) \frac{p(y)}{q(y)} d x d y
$$


So weights are

$$
\beta_i=\frac{p\left(y_i\right)}{q\left(y_i\right)}
$$

Key challenge

We don't know $p(y)$ for the test set.

Because test labels are unknown.

Clever trick: use confusion matrix

Suppose there are $k$ classes.

Build confusion matrix:

$$
C_{i j}=P(\hat{y}=i \mid y=j)
$$


Columns = true label

Rows = predicted label

Then compute

$$
\mu(\hat{y})
$$

= average predicted probabilities on test data.

Then solve

$$
Cp(y)=\mu(\hat{y})
$$


So

$$
p(y)=C^{-1} \mu(\hat{y})
$$


This estimates the test label distribution.

Then weights become

$$
\beta_i=\frac{p\left(y_i\right)}{q\left(y_i\right)}
$$

Why this is nice

Because labels are low dimensional.

Instead of estimating densities in high-dimensional image space, we just solve a small linear system.

#### Concept Shift

Concept shift is much harder.

Because
$p(y \mid x)$ changes

Meaning the definition of the task changes.

Example:

Old task
```
cat vs dog
```

New task
```
white vs black animals
```

Old model becomes useless.

Practical solution

Usually concept shift happens gradually, not abruptly.

Examples in the note:

Advertising

New products appear → click behavior changes.

Cameras

Sensors degrade slowly.

News

Topics evolve gradually.

Strategy: continual learning
Instead of retraining from scratch:
- start from existing model
- update using new data

This is basically online learning / fine-tuning.